# Generalized Linear Models & GDA

[← Back to lesson](https://ml-viz-ruby.vercel.app/courses/linear-regression/04-generalized-linear-models)

This notebook implements three GLMs from scratch (linear regression, logistic regression, Poisson regression), visualizes soft-label distributions, and fits Gaussian Discriminant Analysis — then compares its decision boundary to logistic regression on the same dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748',
    'grid.color': '#2d3748',
    'axes.grid': True,
})

np.random.seed(42)

## 1. The exponential family

We show that Bernoulli, Gaussian, and Poisson distributions all fit the exponential family form:
$$p(y;\eta) = b(y)\exp(\eta T(y) - a(\eta))$$

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

# Show the natural parameter η maps to different link functions
eta = np.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bernoulli: η = log(p/1-p), so p = σ(η)
axes[0].plot(eta, sigmoid(eta), color='#6366f1', linewidth=2)
axes[0].set_title('Bernoulli: p = σ(η)', color='#e2e8f0')
axes[0].set_xlabel('Natural parameter η')
axes[0].set_ylabel('Mean E[y]')

# Gaussian: η = μ (identity link)
axes[1].plot(eta, eta, color='#10b981', linewidth=2)
axes[1].set_title('Gaussian: μ = η (identity link)', color='#e2e8f0')
axes[1].set_xlabel('Natural parameter η')
axes[1].set_ylabel('Mean E[y]')

# Poisson: η = log(λ), so λ = exp(η)
axes[2].plot(eta, np.exp(eta), color='#f59e0b', linewidth=2)
axes[2].set_title('Poisson: λ = exp(η) (log link)', color='#e2e8f0')
axes[2].set_xlabel('Natural parameter η')
axes[2].set_ylabel('Mean E[y]')
axes[2].set_ylim(0, 20)

plt.suptitle('GLM Link Functions: η → E[y]', color='#e2e8f0', y=1.02)
plt.tight_layout()
plt.show()

## 2. GLM gradient update — unified form

For any GLM, the gradient of the log-likelihood is:
$$\nabla_\theta \ell = X^\top(y - \hat{y})$$
This is true for linear regression, logistic regression, and Poisson regression — the same update rule.

In [ ]:
def glm_gradient_descent(X, y, link, n_iters=500, lr=0.1):
    """Generic GLM gradient descent. link maps linear predictor → mean."""
    theta = np.zeros(X.shape[1])
    losses = []
    for _ in range(n_iters):
        eta = X @ theta
        y_hat = link(eta)
        grad = X.T @ (y - y_hat)          # universal GLM gradient
        theta += lr * grad / len(y)
        losses.append(np.mean((y - y_hat)**2))
    return theta, losses

# Poisson regression example: predict count of customer tickets
n = 200
X_raw = np.random.randn(n, 2)
X = np.column_stack([np.ones(n), X_raw])
true_theta = np.array([1.0, 0.8, -0.5])
lam_true = np.exp(X @ true_theta)
y_count = np.random.poisson(lam_true)

theta_pois, losses_pois = glm_gradient_descent(X, y_count, link=np.exp, lr=0.01, n_iters=1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses_pois, color='#f59e0b', linewidth=2)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('MSE')
ax1.set_title('Poisson GLM Training Loss', color='#e2e8f0')

y_pred_pois = np.exp(X @ theta_pois)
ax2.scatter(lam_true, y_pred_pois, alpha=0.4, s=15, color='#f59e0b')
ax2.plot([0, lam_true.max()], [0, lam_true.max()], 'w--', linewidth=1, alpha=0.5)
ax2.set_xlabel('True λ')
ax2.set_ylabel('Predicted λ')
ax2.set_title('Predicted vs True Rate (Poisson GLM)', color='#e2e8f0')
plt.tight_layout()
plt.show()

print(f"True θ:      {true_theta}")
print(f"Fitted θ:    {theta_pois.round(3)}")

## 3. Gaussian Discriminant Analysis (GDA)

GDA models the class-conditional distributions $p(x|y)$ as Gaussians, then applies Bayes' theorem.

In [ ]:
# Generate a 2D binary classification dataset
mu0_true, mu1_true = np.array([-2.0, 0.0]), np.array([2.0, 0.0])
Sigma_true = np.array([[1.5, 0.5], [0.5, 1.0]])

n0, n1 = 100, 100
X0 = np.random.multivariate_normal(mu0_true, Sigma_true, n0)
X1 = np.random.multivariate_normal(mu1_true, Sigma_true, n1)
X_data = np.vstack([X0, X1])
y_data = np.array([0]*n0 + [1]*n1)

# --- GDA fit (closed-form MLE) ---
phi = np.mean(y_data)  # P(y=1)
mu0 = X_data[y_data == 0].mean(axis=0)
mu1 = X_data[y_data == 1].mean(axis=0)
# Pooled covariance (equal covariance assumption)
diff0 = X_data[y_data == 0] - mu0
diff1 = X_data[y_data == 1] - mu1
Sigma_hat = (diff0.T @ diff0 + diff1.T @ diff1) / len(y_data)

def gda_predict_proba(X, phi, mu0, mu1, Sigma):
    """P(y=1|x) via Bayes' theorem with Gaussian class-conditionals."""
    from scipy.stats import multivariate_normal
    log_p0 = multivariate_normal.logpdf(X, mu0, Sigma) + np.log(1 - phi)
    log_p1 = multivariate_normal.logpdf(X, mu1, Sigma) + np.log(phi)
    # log-sum-exp for numerical stability
    log_total = np.logaddexp(log_p0, log_p1)
    return np.exp(log_p1 - log_total)

# --- Logistic regression fit ---
Xb = np.column_stack([np.ones(len(X_data)), X_data])
theta_lr, _ = glm_gradient_descent(Xb, y_data, sigmoid, n_iters=2000, lr=0.5)

# --- Plot both decision boundaries ---
xx, yy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-4, 4, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

proba_gda = gda_predict_proba(grid, phi, mu0, mu1, Sigma_hat).reshape(xx.shape)
Xb_grid = np.column_stack([np.ones(len(grid)), grid])
proba_lr  = sigmoid(Xb_grid @ theta_lr).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, proba, title in zip(axes,
                             [proba_gda, proba_lr],
                             ['GDA Decision Boundary', 'Logistic Regression Boundary']):
    ax.contourf(xx, yy, proba, levels=20, cmap='RdBu', alpha=0.6)
    ax.contour(xx, yy, proba, levels=[0.5], colors='white', linewidths=2)
    ax.scatter(X0[:, 0], X0[:, 1], c='#22d3ee', s=15, alpha=0.7, label='Class 0')
    ax.scatter(X1[:, 0], X1[:, 1], c='#f43f5e', s=15, alpha=0.7, label='Class 1')
    ax.set_title(title, color='#e2e8f0')
    ax.legend()

plt.suptitle('GDA vs Logistic Regression — Both Learn a Linear Boundary', color='#e2e8f0')
plt.tight_layout()
plt.show()

print(f"GDA fitted μ₀: {mu0.round(3)}, μ₁: {mu1.round(3)}")
print(f"GDA fitted Σ diag: {np.diag(Sigma_hat).round(3)}")

## ✏️ Your turn

**Exercise 1 — Quadratic Discriminant Analysis (QDA).** GDA assumes equal covariance $\Sigma$ for both classes. QDA relaxes this, fitting a separate $\Sigma_0$ and $\Sigma_1$ per class. Implement QDA by fitting separate covariances and observe how the decision boundary becomes non-linear (quadratic).

In [ ]:
# TODO(you): implement QDA
# Sigma0 = covariance of class-0 samples only
# Sigma1 = covariance of class-1 samples only
# Then use gda_predict_proba with Sigma0/Sigma1 independently
# Hint: call gda_predict_proba but modify it to accept per-class covariances

In [ ]:
# Assert cell — passes silently when correct
# Fit QDA with separate covariances
Sigma0_ref = (diff0.T @ diff0) / n0
Sigma1_ref = (diff1.T @ diff1) / n1
print("QDA Σ₀ diag:", np.diag(Sigma0_ref).round(3))
print("QDA Σ₁ diag:", np.diag(Sigma1_ref).round(3))
# Both should be close to Sigma_true diag = [1.5, 1.0]
assert abs(np.diag(Sigma0_ref)[0] - 1.5) < 0.5, "Σ₀[0,0] should be ~1.5"

<details><summary>Solution</summary>

```python
from scipy.stats import multivariate_normal

Sigma0 = (diff0.T @ diff0) / n0
Sigma1 = (diff1.T @ diff1) / n1

def qda_predict_proba(X, phi, mu0, mu1, Sigma0, Sigma1):
    log_p0 = multivariate_normal.logpdf(X, mu0, Sigma0) + np.log(1 - phi)
    log_p1 = multivariate_normal.logpdf(X, mu1, Sigma1) + np.log(phi)
    log_total = np.logaddexp(log_p0, log_p1)
    return np.exp(log_p1 - log_total)

proba_qda = qda_predict_proba(grid, phi, mu0, mu1, Sigma0, Sigma1).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, proba_qda, levels=20, cmap='RdBu', alpha=0.6)
plt.contour(xx, yy, proba_qda, levels=[0.5], colors='white', linewidths=2)
plt.scatter(X0[:, 0], X0[:, 1], c='#22d3ee', s=15, alpha=0.7, label='Class 0')
plt.scatter(X1[:, 0], X1[:, 1], c='#f43f5e', s=15, alpha=0.7, label='Class 1')
plt.title('QDA — Quadratic (Non-linear) Boundary')
plt.legend()
plt.show()
```

QDA's decision boundary is quadratic (an ellipse or hyperbola) because different covariances mean the log-likelihood ratio contains quadratic terms in $x$. GDA's equal-covariance assumption causes those terms to cancel, leaving a linear boundary.

</details>